In [1]:
import requests
import os
import time
import json
from bs4 import BeautifulSoup

# ----------- CONFIG -----------
SEARCH_TERM = "Congressional Record"
YEARS = list(range(1930, 1951))
RESULTS_PER_YEAR = 100
SAVE_DIR = "ocr_output"
os.makedirs(SAVE_DIR, exist_ok=True)

# ----------- STEP 1: GET HTIDs -----------
def get_htids_for_year(year):
    htids = []
    url = f"https://catalog.hathitrust.org/api/volumes/brief/json/search?title={SEARCH_TERM.replace(' ', '+')}&year={year}&limit={RESULTS_PER_YEAR}"
    resp = requests.get(url)
    if resp.status_code != 200:
        print(f"Failed to fetch volumes for {year}")
        return htids
    data = resp.json()
    for key in data.get('records', {}):
        for item in data['records'][key]['items']:
            htids.append(item['htid'])
    return htids

# ----------- STEP 2: DOWNLOAD OCR TEXT -----------
def download_ocr_for_htid(htid):
    output_file = os.path.join(SAVE_DIR, f"{htid.replace('/', '_')}.txt")
    if os.path.exists(output_file):
        print(f"Skipping {htid}, already downloaded.")
        return

    url = f"https://babel.hathitrust.org/cgi/pt?id={htid};view=1up;seq=1"
    base_url = f"https://catalog.hathitrust.org/api/volumes/full/htid/{htid}.json"

    try:
        response = requests.get(base_url)
        if response.status_code != 200:
            print(f"Failed to get metadata for {htid}")
            return
        vol_data = response.json()
        pages = vol_data.get('items', [])

        # You could scrape OCR page-by-page from page URLs (deprecated way)
        ocr_text = []
        for seq_num in range(1, 20):  # LIMIT FOR DEMO. REMOVE TO GET FULL VOLUME
            page_url = f"https://babel.hathitrust.org/cgi/imgsrv/pt?id={htid};seq={seq_num};view=ocr"
            r = requests.get(page_url)
            if r.status_code != 200:
                break
            soup = BeautifulSoup(r.text, "html.parser")
            text = soup.get_text()
            ocr_text.append(text.strip())
            time.sleep(0.5)  # Be polite

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("\n".join(ocr_text))
            print(f"Saved OCR for {htid}")

    except Exception as e:
        print(f"Error fetching OCR for {htid}: {e}")

# ----------- MAIN PIPELINE -----------
if __name__ == "__main__":
    for year in YEARS:
        print(f"Processing year {year}")
        htids = get_htids_for_year(year)
        for htid in htids:
            download_ocr_for_htid(htid)
            time.sleep(1)  # To avoid hammering servers


Processing year 1930
Failed to fetch volumes for 1930
Processing year 1931
Failed to fetch volumes for 1931
Processing year 1932
Failed to fetch volumes for 1932
Processing year 1933
Failed to fetch volumes for 1933
Processing year 1934
Failed to fetch volumes for 1934
Processing year 1935
Failed to fetch volumes for 1935
Processing year 1936
Failed to fetch volumes for 1936
Processing year 1937
Failed to fetch volumes for 1937
Processing year 1938
Failed to fetch volumes for 1938
Processing year 1939
Failed to fetch volumes for 1939
Processing year 1940
Failed to fetch volumes for 1940
Processing year 1941
Failed to fetch volumes for 1941
Processing year 1942
Failed to fetch volumes for 1942
Processing year 1943
Failed to fetch volumes for 1943
Processing year 1944
Failed to fetch volumes for 1944
Processing year 1945
Failed to fetch volumes for 1945
Processing year 1946
Failed to fetch volumes for 1946
Processing year 1947
Failed to fetch volumes for 1947
Processing year 1948
Failed 